## Example 1: Basic Encoding Detection

In [ ]:
import charset_normalizer

# Example 1: Basic encoding detection and text normalization
print("Example 1: Testing basic encoding detection")

try:
    # Test with various text samples in different encodings
    test_texts = [
        b"Hello, World!",  # ASCII
        "Hello, World! \u00e9\u00f1\u00fc".encode('utf-8'),  # UTF-8 with accents
        "Hello, World! \u00e9\u00f1\u00fc".encode('latin1'),  # Latin-1
        "\u4f60\u597d\u4e16\u754c".encode('utf-8'),  # Chinese in UTF-8
        "\u0417\u0434\u0440\u0430\u0432\u0441\u0442\u0432\u0443\u0439".encode('utf-8')  # Russian in UTF-8
    ]
    
    print("Detecting encoding for various text samples:")
    for i, sample in enumerate(test_texts, 1):
        # Detect encoding using charset_normalizer
        detection_result = charset_normalizer.detect(sample)
        print(f"  Sample {i}: {detection_result['encoding']} (confidence: {detection_result['confidence']:.2f})")
        
        # Normalize the text
        normalized = str(charset_normalizer.from_bytes(sample).best())
        print(f"    Text: {normalized[:30]}{'...' if len(normalized) > 30 else ''}")
    
    print("✓ Basic encoding detection working")
    print("charset_normalizer example 1 completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"charset_normalizer example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 1 finished")

## Example 2: Advanced Detection with Multiple Results

In [ ]:
import charset_normalizer

# Example 2: Advanced detection with multiple encoding candidates
print("Example 2: Testing advanced detection with multiple results")

try:
    # Create ambiguous text that could match multiple encodings
    ambiguous_text = "Café Rösnerß Brüderschaft".encode('latin1')
    
    print("Testing detection with potentially ambiguous encoding:")
    
    # Get all possible encoding results
    results = charset_normalizer.from_bytes(ambiguous_text)
    
    print(f"Number of encoding candidates found: {len(results)}")
    
    # Display top candidates
    candidate_count = 0
    for candidate in results:
        candidate_count += 1
        if candidate_count > 3:
            break
        print(f"  Candidate {candidate_count}: {candidate.encoding}")
        
        # Access confidence score safely
        try:
            confidence = float(candidate.fingerprint) if hasattr(candidate, 'fingerprint') else 0.0
            print(f"    Confidence: {confidence:.2f}")
        except (TypeError, ValueError):
            print(f"    Confidence: N/A")
        
        # Get text safely
        try:
            text = str(candidate)
            print(f"    Text: {text[:40]}{'...' if len(text) > 40 else ''}")
        except Exception:
            print(f"    Text: [Unable to decode]")
            raise
    
    # Get the best result
    best_result = results.best()
    if best_result:
        print(f"\nBest encoding: {best_result.encoding}")
        print(f"Best result text: {str(best_result)}")
    else:
        print("\nNo best encoding found")
    
    # Test with raw bytes detection
    raw_detection = charset_normalizer.detect(ambiguous_text)
    print(f"\nRaw detection result:")
    print(f"  Encoding: {raw_detection.get('encoding', 'Unknown')}")
    print(f"  Confidence: {raw_detection.get('confidence', 0.0):.2f}")
    
    print("✓ Advanced detection with multiple results working")
    print("charset_normalizer example 2 completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Runtime error: {e}")
    import traceback
    traceback.print_exc()
    raise
    
print("Example 2 finished")

## Example 3: File Encoding Detection and Normalization

In [ ]:
import charset_normalizer
import tempfile
import os

# Example 3: File encoding detection and text normalization
print("Example 3: Testing file encoding detection")

try:
    # Create temporary files with different encodings
    test_content = "Hello, World! Caf\u00e9 R\u00f6sti \u4f60\u597d"
    
    encodings_to_test = ['utf-8', 'latin1', 'cp1252', 'iso-8859-1']
    temp_files = []
    
    print("Creating test files with different encodings:")
    for encoding in encodings_to_test:
        try:
            # Create temporary file
            temp_fd, temp_path = tempfile.mkstemp(suffix=f'_{encoding}.txt')
            temp_files.append(temp_path)
            
            # Write content with specific encoding
            with os.fdopen(temp_fd, 'wb') as f:
                f.write(test_content.encode(encoding, errors='ignore'))
            
            print(f"  Created file with {encoding} encoding: {os.path.basename(temp_path)}")
            
        except (UnicodeEncodeError, LookupError) as e:
            print(f"  Skipped {encoding}: {e}")
            if temp_path in temp_files:
                temp_files.remove(temp_path)
    
    print("\nDetecting encoding for each file:")
    for file_path in temp_files:
        try:
            # Detect encoding from file
            detection_result = charset_normalizer.from_path(file_path)
            best = detection_result.best()
            
            print(f"  File {os.path.basename(file_path)}:")
            print(f"    Detected: {best.encoding if best else 'Unknown'}")
            print(f"    Content: {str(best)[:50] if best else 'N/A'}{'...' if best and len(str(best)) > 50 else ''}")
            
        except Exception as e:
            print(f"    Error detecting {file_path}: {e}")
            raise
    
    # Cleanup
    for file_path in temp_files:
        try:
            os.unlink(file_path)
        except OSError:
            pass
    
    print("✓ File encoding detection working")
    print("charset_normalizer example 3 completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"charset_normalizer example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 3 finished")

## Example 4: Encoding Comparison and Validation

In [ ]:
import charset_normalizer

# Example 4: Encoding comparison and validation
print("Example 4: Testing encoding comparison and validation")

try:
    # Test different text samples that might be problematic
    test_cases = [
        ("Simple ASCII", b"Hello World"),
        ("UTF-8 with BOM", b"\xef\xbb\xbfHello World"),
        ("Latin-1 accented", "Café Rösti".encode('latin1')),
        ("Mixed content", b"ASCII + \xc3\xa9 + more"),
        ("Empty content", b""),
        ("Binary-like data", bytes(range(256))[:50])
    ]
    
    print("Testing encoding detection confidence and validation:")
    
    for name, data in test_cases:
        print(f"\n  {name}:")
        
        # Basic detection
        basic_result = charset_normalizer.detect(data)
        encoding = basic_result.get('encoding', 'Unknown')
        confidence = basic_result.get('confidence', 0.0)
        print(f"    Basic detection: {encoding} ({confidence:.2f})")
        
        # Advanced detection with multiple candidates
        advanced_results = charset_normalizer.from_bytes(data)
        print(f"    Candidates found: {len(advanced_results)}")
        
        if advanced_results:
            best = advanced_results.best()
            if best:
                # Safely get fingerprint as float
                try:
                    fingerprint = float(best.fingerprint) if hasattr(best, 'fingerprint') else 0.0
                except (TypeError, ValueError):
                    fingerprint = 0.0
                    
                print(f"    Best candidate: {best.encoding} (fingerprint: {fingerprint:.2f})")
                
                # Show decoded content if it looks like text
                if fingerprint > 0.1:
                    content = str(best)[:30]
                    print(f"    Content preview: {repr(content)}")
                else:
                    print(f"    Content: [Binary or low confidence]")
        
        # Test validation: the bytes must round-trip through the detected encoding. Measured (3.5.1): every sample gets
        # an encoding and round-trips; the 10-byte Latin-1 sample is detected as cp1006, a single-byte codec, so it does too
        assert encoding and encoding != 'Unknown', f"{name}: no encoding detected: {basic_result}"
        decoded = data.decode(encoding)
        reencoded = decoded.encode(encoding)
        assert reencoded == data, f"{name}: round-trip through {encoding} changed the bytes: {reencoded!r} != {data!r}"
        print("    Validation: ✓ Valid")
    
    print("\n✓ Encoding comparison and validation working")
    print("charset_normalizer example 4 completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Runtime error: {e}")
    import traceback
    traceback.print_exc()
    raise
    
print("Example 4 finished")

## Example 5: Production Text Processing Pipeline

In [ ]:
import charset_normalizer
import time

# Example 5: Production text processing pipeline
print("Example 5: charset_normalizer production text processing")

try:
    # Simulate a text processing pipeline
    class TextProcessor:
        def __init__(self, min_confidence=0.7):
            self.min_confidence = min_confidence
            self.stats = {
                'processed': 0,
                'successful': 0,
                'failed': 0,
                'encodings': {},
                'total_time': 0
            }
        
        def process_data(self, data, name="unknown"):
            """Process raw bytes and extract normalized text"""
            start_time = time.time()
            
            try:
                # Detect encoding with charset_normalizer
                results = charset_normalizer.from_bytes(data)
                
                if not results:
                    self.stats['failed'] += 1
                    return None, "No encoding detected"
                
                best = results.best()
                
                # Confidence = 1 - chaos (the library's mess ratio). NOT `fingerprint`: in 3.5.1 that is hash(str(match)),
                # random per process (measured), which made this pipeline's accept/reject verdicts random run to run
                confidence = 1.0 - best.chaos
                
                if confidence < self.min_confidence:
                    self.stats['failed'] += 1
                    return None, f"Low confidence: {confidence:.2f}"
                
                # Extract normalized text
                text = str(best)
                encoding = best.encoding
                
                # Update statistics
                self.stats['successful'] += 1
                self.stats['encodings'][encoding] = self.stats['encodings'].get(encoding, 0) + 1
                
                return {
                    'name': name,
                    'encoding': encoding,
                    'confidence': confidence,
                    'text': text,
                    'length': len(text),
                    'candidates': len(results)
                }, None
                
            except Exception as e:
                self.stats['failed'] += 1
                return None, str(e)
            
            finally:
                self.stats['processed'] += 1
                self.stats['total_time'] += time.time() - start_time
        
        def get_statistics(self):
            """Get processing statistics"""
            return self.stats.copy()
    
    # Create processor and test data
    processor = TextProcessor(min_confidence=0.5)
    
    # Test with various data samples
    test_samples = [
        ("english.txt", b"Hello, World! This is English text."),
        ("french.txt", "Bonjour! Café crème s'il vous plaît.".encode('utf-8')),
        ("german.txt", "Guten Tag! Schön Sie zu sehen, Herr Müller.".encode('utf-8')),
        ("spanish.txt", "¡Hola! ¿Cómo está usted? Muy bien, gracias.".encode('latin1')),
        ("chinese.txt", "你好世界！欢迎来到中国。".encode('utf-8')),
        ("russian.txt", "Привет, мир! Как дела?".encode('utf-8')),
        ("mixed.txt", b"ASCII + \xc3\xa9\xc3\xb1\xc3\xbc UTF-8 + more content"),
        ("binary.dat", bytes(range(50)))
    ]
    
    print("Processing text samples:")
    results = []
    # the text samples' source strings: detection must recover every one of them exactly
    source_text = {name: data.decode('latin1' if name == 'spanish.txt' else 'utf-8') for name, data in test_samples if name != 'binary.dat'}
    
    for name, data in test_samples:
        result, error = processor.process_data(data, name)
        
        if result:
            print(f"  ✓ {result['name']}: {result['encoding']} ({result['confidence']:.2f})")
            print(f"    Text: {result['text'][:40]}{'...' if result['length'] > 40 else ''}")
            results.append(result)
        else:
            print(f"  ✗ {name}: {error}")
        if name in source_text:
            assert result and result['text'] == source_text[name], (name, error or result['encoding'])
    
    # Show processing statistics
    stats = processor.get_statistics()
    print(f"\nProcessing Statistics:")
    print(f"  Total processed: {stats['processed']}")
    print(f"  Successful: {stats['successful']}")
    print(f"  Failed: {stats['failed']}")
    # measured (3.5.1): nothing is refused - binary.dat's bytes 0..49 are valid ASCII (utf_8, chaos 0.2 -> confidence 0.8)
    assert stats['failed'] == 0 and stats['successful'] == len(test_samples), stats
    
    if stats['processed'] > 0:
        success_rate = stats['successful']/stats['processed']*100
        avg_time = stats['total_time']/stats['processed']*1000
        print(f"  Success rate: {success_rate:.1f}%")
        print(f"  Average time per sample: {avg_time:.2f}ms")
    
    if stats['encodings']:
        print(f"  Encoding distribution: {dict(list(stats['encodings'].items())[:5])}")
    
    print("\nProduction pipeline capabilities:")
    capabilities = [
        "🔍 Automatic encoding detection with confidence scoring",
        "📝 Text normalization and extraction from raw bytes",
        "⚡ High-performance batch processing of multiple files",
        "📊 Comprehensive statistics and monitoring",
        "🛡️ Error handling and fallback strategies",
        "🌐 Support for international text and multiple encodings"
    ]
    
    for capability in capabilities:
        print(f"  {capability}")
    
    print("\n✓ Production text processing pipeline working")
    print("charset_normalizer example 5 completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Runtime error: {e}")
    import traceback
    traceback.print_exc()
    raise
    
print("Example 5 finished")